In [1]:
import os
import random
import string
import hail as hl
from ukb_utils import hail_init
import numpy as np
from ko_utils import ko
from ko_utils import samples
from ko_utils.ko import PLOF_CSQS, MISSENSE_CSQS, SYNONYMOUS_CSQS, OTHER_CSQS

In [2]:
2

2

In [3]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compa030.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [3]:




def count_phased_calls(gt, seed = None):
    """ Randomize genotype phase of call
    
    :param gt: genotype call to be flipped
    """
    assert str(gt.dtype) == 'call'
    boolean = hl.rand_bool(0.5, seed)
    return((hl.case()
    .when(gt.n_alt_alleles() != 1, gt)
    .when(boolean == 1, hl.parse_call("0|1"))
    .when(boolean == 0, hl.parse_call("1|0"))
    .or_missing()))



In [19]:
calls = mt.GT

calls.filter #calls.n_alt_alleles() == 1

AttributeError: 'CallExpression' object has no attribute 'filter'

In [ ]:
def set_to_phased_call(gt):
    """ Set an unphased genotype call to phased genotype 
    
    :param gt: genotype call to be converted
    """
    assert str(gt.dtype) == 'call'
    return((hl.case()
    .when(gt == hl.parse_call("1/0"), hl.parse_call("1|0"))
    .when(gt == hl.parse_call("0/1"), hl.parse_call("0|1"))
    .when(gt == hl.parse_call("1/1"), hl.parse_call("1|1"))
    .when(gt == hl.parse_call("0/0"), hl.parse_call("0|0"))
    .or_missing()))

In [ ]:
# identify individuals who are compound hetz

In [ ]:
mt = hl.read_matrix_table('data/simulation/data/ukb_eur_10000_samples_chr21.mt')
mt = mt.explode_rows(mt.consequence.vep.worst_csq_by_gene_canonical)
mt = mt.annotate_rows(
    consequence_category=ko.csqs_case_builder(
            worst_csq_expr=mt.consequence.vep.worst_csq_by_gene_canonical,
            use_loftee=True)
        )

# subset to current csqs category
items = ['pLoF','LC','damaging_missense']
mt_damaging = mt.filter_rows(hl.literal(set(items)).contains(mt.consequence_category))


# convert to gene x sample matrix
mt_gene = (mt_damaging.group_rows_by(mt_damaging.consequence.vep.worst_csq_by_gene_canonical.gene_id)
        .aggregate(gts=hl.agg.collect(mt_damaging.GT)))

# determine genes that are knocked out
mt_gene = ko.sum_gts_entries(mt_gene)
expr_pko = ko.calc_prob_ko(mt_gene.hom_alt, mt_gene.phased, mt_gene.unphased)
expr_ko = ko.annotate_knockout(mt_gene.hom_alt, expr_pko)
mt_gene = mt_gene.annotate_entries(
        pKO = expr_pko,
        knockout = expr_ko)

In [ ]:
# what genes should we continue with?
ht = mt_gene.filter_entries(mt_gene.knockout == 'Compound heterozygote').entries()
ht.aggregate(hl.agg.counter(ht.gene_id))

In [ ]:
# what individuals should we continue with
ht = mt_gene.filter_entries(mt_gene.knockout == 'Compound heterozygote').entries()
samples = mt.s.collect()
samples_ch = ht.s.collect()
samples_not_ch = set(samples) - set(samples_ch)

In [ ]:
# what samples to select
n = 100
sids = samples[0:(n-len(samples_ch))]
sids = (set(sids) | set(samples_ch))

In [ ]:
# what gene to select
mt = mt.filter_rows(hl.literal("ENSG00000160183").contains(mt.consequence.vep.worst_csq_by_gene_canonical.gene_id))
mt = mt.filter_cols(hl.literal(sids).contains(mt.s))
mt.count()


In [80]:
checkpoint = 'data/simulation/checkpoint/chr21_v100_s352_8ch.mt'
mt.write(checkpoint)

2022-02-25 14:06:17 Hail: INFO: wrote matrix table with 352 rows and 100 columns in 1 partition to data/simulation/checkpoint/chr21_v100_s352_8ch.mt


In [4]:
checkpoint = 'data/simulation/checkpoint/chr21_v100_s352_8ch.mt'
mt = hl.read_matrix_table(checkpoint)

In [5]:
ko.agg_count_calls(mt)

(44, 39)

In [80]:


mt = hl.read_matrix_table(checkpoint)
tmp = mt.transmute_entries(GT = ko.rand_flip_call(mt.GT))
d = ko.agg_count_calls(tmp)
d

frozendict({Call(alleles=[0, 1], phased=True): 44, Call(alleles=[1, 0], phased=True): 16})

Call(alleles=[0, 1], phased=True)

In [31]:
#mt = mt.drop('consequence') #mt.consequence_category.collect()
items = ['pLoF','LC','damaging_missense']
mt_damaging = mt.filter_rows(hl.literal(set(items)).contains(mt.consequence_category))


# convert to gene x sample matrix
mt_gene = (mt_damaging.group_rows_by(mt_damaging.consequence.vep.worst_csq_by_gene_canonical.gene_id)
        .aggregate(gts=hl.agg.collect(mt_damaging.GT)))

# determine genes that are knocked out
mt_gene = ko.sum_gts_entries(mt_gene)
expr_pko = ko.calc_prob_ko(mt_gene.hom_alt, mt_gene.phased, mt_gene.unphased)
expr_ko = ko.annotate_knockout(mt_gene.hom_alt, expr_pko)
mt_gene = mt_gene.annotate_entries(
        pKO = expr_pko,
        knockout = expr_ko)

In [24]:
ht = mt_gene.filter_entries(mt_gene.knockout == 'Compound heterozygote').entries()
M = ht.count()
ht = ht.annotate(beta=hl.rand_bool(1) * hl.rand_norm(0, hl.sqrt(h2 / (M))))

2022-03-01 11:24:31 Hail: INFO: Coerced sorted dataset


In [25]:
ht

In [5]:


mt = hl.experimental.ldscsim.make_betas(mt, 0.9, pi = 0.1)[0]

In [6]:
betas = mt.beta.collect()

In [7]:
mt = hl.experimental.ldscsim.normalize_genotypes(mt.GT)

In [8]:
def _get_tid(length=5):
    r"""Internal method for getting random ID string to append to field names"""
    return ''.join(random.choices(string.ascii_uppercase + string.ascii_lowercase, 
                                  k=length))

def annotate_all(mt, row_exprs={}, col_exprs={}, entry_exprs={}, global_exprs={}):
    r"""Equivalent of _annotate_all, but checks source MatrixTable of exprs"""
    exprs = {**row_exprs, **col_exprs, **entry_exprs, **global_exprs}
    for key, value in exprs.items():
        if type(value) == hl.expr_float64 or type(value) == hl.expr_int32:
            assert value._indices.source == mt, 'Cannot combine expressions from different source objects.'
    return mt._annotate_all(row_exprs, col_exprs, entry_exprs, global_exprs)

In [9]:
beta = mt.beta
popstrat=None
popstrat_var=None
genotype = mt.GT
h2 = 0.9

In [10]:
tid = _get_tid()
mt = annotate_all(mt=mt,
                 row_exprs={'beta_'+tid: beta},
                 col_exprs={} if popstrat is None else {'popstrat_'+tid: popstrat},
                 entry_exprs={'gt_'+tid: genotype.n_alt_alleles() if genotype.dtype is hl.dtype('call') else genotype})
mt = mt.filter_rows(hl.agg.stats(mt['gt_'+tid]).stdev>0)
mt = hl.experimental.ldscsim.normalize_genotypes(mt['gt_'+tid])


In [11]:
mt = mt.annotate_cols(y_no_noise=hl.agg.sum(mt['beta_'+tid] * mt['norm_gt']))
#mt.aggregate_cols(hl.agg.mean(mt.y_no_noise))
mt = mt.annotate_cols(y=mt.y_no_noise + hl.rand_norm(0, hl.sqrt(1-h2)))

In [13]:
path = 'data/permute/counts/ukb_eur_wes_200k_pLoF_damaging_missense_counts_chr21.vcf.gz'


<Float64Expression of type float64>

In [12]:
#result_ht = hl.linear_regression_rows(
 #    y=mt.y,
 #    x=mt.GT.n_alt_alleles(),
 #    covariates=[1])

In [195]:
def rand_flip_call(gt: hl.call, P: float = 0.5, seed = None):
    """ Randomize genotype phase of call

    :param gt: genotype call to be flipped
    :param P: probabily of one phase
    :param seed: seed for random
    """
    assert str(gt.dtype) == 'call'
    return hl.if_else(
        (gt.n_alt_alleles() == 1) &
        (is_phased(gt)),
        hl.if_else(
            hl.rand_bool(P),
            hl.parse_call("1|0"),
            hl.parse_call("0|1")
        ),
        gt
    )
    
    #return((hl.case()
    #.when(gt.n_alt_alleles() != 1, gt)
    #.when(boolean == True, hl.parse_call("0|1"))
    #.when(boolean == False, hl.parse_call("1|0"))
    #.or_missing()))

In [3]:
def agg_count_calls(mt: hl.MatrixTable, phased: bool = True):
    """ Count number of phased/unphased hetz and what haplotype they reside on

    :param mt: MatrixTable with GT field
    :param mt: test for phased genotypes only
    """
    aggr = mt.aggregate_entries(hl.agg.counter(mt.GT))
    gt10 = aggr[hl.Call(alleles=[1, 0], phased = phased)]
    gt01 = aggr[hl.Call(alleles=[0, 1], phased = phased)]
    return((gt10,gt01))
    

# manually simulated with Hail

In [4]:
n = 10
mt = hl.balding_nichols_model(10, 100, 1000, reference_genome='GRCh38')
mt = mt.transmute_entries(GT = ko.set_to_phased_call(mt.GT))
mt = mt.transmute_entries(GT = ko.rand_flip_call(mt.GT))
ko.agg_count_calls(mt)

2022-03-01 12:46:30 Hail: INFO: balding_nichols_model: generating genotypes for 10 populations, 100 samples, and 1000 variants...


(17799, 17574)

In [128]:
d[hl.Call(alleles=[0, 0], phased=True)]

31050

In [4]:
mt.aggregate_entries(hl.agg.sum(ko.is_phased(mt.GT)))

100000

In [78]:
n = 10
mt = hl.balding_nichols_model(1, 10, n, reference_genome='GRCh38')
mt = mt.transmute_entries(GT = ko.set_to_phased_call(mt.GT))
mt = hl.experimental.ldscsim.simulate_phenotypes(mt, mt.GT[0], h2 = 1)
mt = mt.annotate_rows(maf = hl.min(hl.agg.call_stats(mt.GT, mt.alleles).AF))
mt = mt.annotate_rows(ES = 2 * (mt.beta**2)*mt.maf*(1-mt.maf))
#mt = mt.annotate_rows(h2i = mt.ES * mt.n)
#sum(mt.gv[0].collect())

2022-02-24 22:28:05 Hail: INFO: balding_nichols_model: generating genotypes for 1 populations, 10 samples, and 10 variants...


calculating phenotype


In [79]:
mt.show()

2022-02-24 22:28:07 Hail: INFO: Coerced sorted dataset


+---------------+------------+------+-----------+------+-----------+------+-----------+------+-----------+------+-----------+
| locus         | alleles    | 0.GT | 0.norm_gt | 1.GT | 1.norm_gt | 2.GT | 2.norm_gt | 3.GT | 3.norm_gt | 4.GT | 4.norm_gt |
+---------------+------------+------+-----------+------+-----------+------+-----------+------+-----------+------+-----------+
| locus<GRCh38> | array<str> | call |   float64 | call |   float64 | call |   float64 | call |   float64 | call |   float64 |
+---------------+------------+------+-----------+------+-----------+------+-----------+------+-----------+------+-----------+
| chr1:1        | ["A","C"]  | 1|0  |  8.16e-01 | 0|0  | -1.22e+00 | 0|0  | -1.22e+00 | 1|0  |  8.16e-01 | 1|0  |  8.16e-01 |
| chr1:2        | ["A","C"]  | 1|0  |  3.33e-01 | 0|0  | -3.00e+00 | 1|0  |  3.33e-01 | 1|0  |  3.33e-01 | 1|0  |  3.33e-01 |
| chr1:7        | ["A","C"]  | 1|0  |  6.55e-01 | 0|0  | -1.53e+00 | 1|1  |  6.55e-01 | 1|0  |  6.55e-01 | 0|0  | -1.53e+00 |
| chr1:10       | ["A","C"]  | 0|0  | -3.00e+00 | 1|0  |  3.33e-01 | 1|0  |  3.33e-01 | 1|1  |  3.33e-01 | 1|1  |  3.33e-01 |
+---------------+------------+------+-----------+------+-----------+------+-----------+------+-----------+------+-----------+
showing the first 5 of 10 columns

In [81]:
mt.y.show()

,
sample_idx,y
int32,float64
0,1.36e-01
1,3.57e-01
2,4.65e-01
3,-5.39e-03
4,-9.36e-01
5,-4.66e-01
6,-5.39e-03
7,-5.39e-03


In [513]:
annotation = ["pLoF","LC","damaging_missense","other_missense","synonymous"]
weights = [0.02, 0.03, 0.05, 0.1, 0.8]
def random_gene_annotation(k, annotation = ['a','b'], weights = [0.4, 0.6]):
    return(random.choices(annotation, weights = weights, k = k))

In [514]:
df = mt.rows().drop('bn').to_pandas()
df[['csqs']] = random_annotation(n, annotation, weights)
df[['gene_id']] = 'CACNA1C'

2022-02-23 18:15:21 Hail: INFO: Coerced sorted dataset


In [515]:

ht = hl.Table.from_pandas(df)
ht = ht.annotate(
    locus = hl.parse_variant(
        hl.delimit([ht['locus.contig'],hl.str(ht['locus.position']),ht.alleles[0],ht.alleles[1]],':')
    )
)
ht = ht.key_by(ht.locus)
mt = mt.annotate_rows(
    gene_id = ht[mt.row_key].gene_id,
    csqs = ht[mt.row_key].csqs

)
mt = mt.filter_rows(hl.literal(set(['pLoF','LC','damaging_missense'])).contains(mt.csqs))
mt.count()

2022-02-23 18:15:23 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:23 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:23 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:24 Hail: INFO: Ordering unsorted dataset with network shuffle


(108, 10)

In [516]:
mt_gene = (mt.group_rows_by(mt.gene_id)
            .aggregate(gts=hl.agg.collect(mt.GT)))

In [517]:
mt_gene = ko.sum_gts_entries(mt_gene)
expr_pko = ko.calc_prob_ko(mt_gene.hom_alt, mt_gene.phased, mt_gene.unphased)
expr_ko = ko.annotate_knockout(mt_gene.hom_alt, expr_pko)
mt_gene = mt_gene.annotate_entries(
            pKO = expr_pko,
            knockout = expr_ko)
mt_gene.knockout.show()

2022-02-23 18:15:31 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:31 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:32 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:33 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-02-23 18:15:33 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:33 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:34 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-02-23 18:15:36 Hail: INFO: Coerced sorted dataset


,,,,,
,0,1,2,3,4
gene_id,knockout,knockout,knockout,knockout,knockout
str,str,str,str,str,str
"""CACNA1C""","""Homozygote""","""Homozygote""","""Homozygote""","""Homozygote""","""Homozygote"""


(182, 500)

In [273]:
mt_gene = (mt.group_rows_by(mt.consequence.vep.worst_csq_by_gene_canonical.gene_id)
            .aggregate(gts=hl.agg.collect(mt.GT)))

In [275]:
mt_gene = ko.sum_gts_entries(mt_gene)
expr_pko = ko.calc_prob_ko(mt_gene.hom_alt, mt_gene.phased, mt_gene.unphased)
expr_ko = ko.annotate_knockout(mt_gene.hom_alt, expr_pko)
mt_gene = mt_gene.annotate_entries(
            pKO = expr_pko,
            knockout = expr_ko)

In [508]:
mt_gene.pKO.show()

2022-02-23 18:10:40 Hail: INFO: Coerced sorted dataset
2022-02-23 18:10:40 Hail: INFO: Coerced sorted dataset
2022-02-23 18:10:41 Hail: INFO: Coerced sorted dataset
2022-02-23 18:10:41 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-02-23 18:10:42 Hail: INFO: Coerced sorted dataset


,,,,,
,0,1,2,3,4
gene_id,pKO,pKO,pKO,pKO,pKO
str,float64,float64,float64,float64,float64
"""CACNA1C""",1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.00e+00


# Simulating matrix appending

In [28]:


#

2022-02-26 23:25:57 Hail: INFO: balding_nichols_model: generating genotypes for 1 populations, 2 samples, and 18 variants...


In [29]:
mts[0].show()

2022-02-26 23:25:58 Hail: INFO: Coerced sorted dataset


,,,
,,0,1
locus,alleles,GT,GT
locus<GRCh38>,array<str>,call,call
chr1:1,"[""A"",""C""]",0/0,0/1
chr1:2,"[""A"",""C""]",1/1,1/1
chr1:3,"[""A"",""C""]",0/0,0/1
chr1:4,"[""A"",""C""]",1/1,1/1
chr1:5,"[""A"",""C""]",1/1,1/1
chr1:6,"[""A"",""C""]",0/1,0/0
chr1:7,"[""A"",""C""]",1/1,0/1


In [63]:
import os

os.listdir('data/phased/wes_union_calls/ukb_wes_union_calls_200k_chr21-24xshort.qa/')

['prs52000_pro13000_mprs100000.1of1.vcf.gz',
 'prs52000_pro13000_mprs100000.1of.log',
 'prs52000_pro13000_mprs100000.1of1.txt',
 'prs52000_pro13000_mprs100000.1of1.trio']

In [57]:
import math

lst = ['aaaabbbc','dddeeefff','geegghhhi']
[l for l in lst if 'ee' in l]


['dddeeefff', 'geegghhhi']

In [55]:
mt = hl.balding_nichols_model(1, 2, 18, reference_genome='GRCh38')
mts = [mt.filter_rows((mt.locus.position > i-3) & (mt.locus.position <= i+8)) for i in range(0,18,6)]

if len(mts) > 1:
    
    i = 0
    max_iter = len(mts)-1
    mt1 = mts[0]
    mt2 = mts[1]
    final = list()
    while (len(final) < max_iter):

        # get overlapping fraction
        overlap = mt1.filter_rows(hl.is_defined(mt2.rows()[mt1.row_key]))
        n_overlap = overlap.count()[0]

        # append indexes to overlap, and select right flank for mt1 and left flank for mt2
        overlap = overlap.add_row_index()
        mt1_right_flank = overlap.filter_rows(overlap.row_idx == math.floor(n_overlap / 2)).locus.position.collect()
        mt2_left_flank = overlap.filter_rows(overlap.row_idx == math.ceil(n_overlap / 2)).locus.position.collect()
        print(f"iter={i},mt1_right_flank={mt1_right_flank},mt2_left_flank={mt2_left_flank}")
            
        # Trim right flank.
        mt1 = mt1.filter_rows(mt1.locus.position <= mt1_right_flank[0])

        # if mt1 is the first matrix (from left to right) it is now done.
        # otherwise continue and cut down left flank.
        #iif i == 0:
        final.append(mt1)

        # Finish left flank.
        # if mt2 is the last matrix, it's now done.
        # if mt2 is the 2nd matrix, the left flank is now done. Set mt2 to mt1
        # in order to trim its other flank
        mt2 = mt2.filter_rows(mt2.locus.position >= mt2_left_flank[0])

        i += 1
        mt1 = mt2
        if (i < max_iter):
            mt2 = mts[i+1]

    # add right-most matrix table
    final.append(mt2)
    mt = hl.MatrixTable.union_rows(*final)
    
    
mt.show()

2022-02-26 23:36:50 Hail: INFO: balding_nichols_model: generating genotypes for 1 populations, 2 samples, and 18 variants...
2022-02-26 23:36:50 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:50 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:51 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:51 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:51 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:51 Hail: INFO: Coerced sorted dataset


iter=0,mt1_right_flank=[6],mt2_left_flank=[7]


2022-02-26 23:36:52 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:52 Hail: INFO: reading 6 of 8 data partitions
2022-02-26 23:36:52 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:53 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:53 Hail: INFO: reading 6 of 8 data partitions
2022-02-26 23:36:53 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:54 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:54 Hail: INFO: reading 6 of 8 data partitions
2022-02-26 23:36:54 Hail: INFO: Coerced sorted dataset


iter=1,mt1_right_flank=[12],mt2_left_flank=[13]


2022-02-26 23:36:54 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:55 Hail: INFO: reading 2 of 8 data partitions
2022-02-26 23:36:55 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:55 Hail: INFO: reading 3 of 8 data partitions
2022-02-26 23:36:55 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:55 Hail: INFO: reading 2 of 8 data partitions
2022-02-26 23:36:56 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:56 Hail: INFO: reading 2 of 8 data partitions
2022-02-26 23:36:56 Hail: INFO: Coerced sorted dataset
2022-02-26 23:36:56 Hail: INFO: reading 3 of 8 data partitions
2022-02-26 23:36:56 Hail: INFO: Coerced sorted dataset


,,,
,,0,1
locus,alleles,GT,GT
locus<GRCh38>,array<str>,call,call
chr1:1,"[""A"",""C""]",0/1,0/0
chr1:2,"[""A"",""C""]",1/1,1/1
chr1:3,"[""A"",""C""]",0/1,0/1
chr1:4,"[""A"",""C""]",0/1,1/1
chr1:5,"[""A"",""C""]",1/1,1/1
chr1:6,"[""A"",""C""]",0/0,0/0
chr1:7,"[""A"",""C""]",0/1,0/0


In [53]:
mt.locus.position.collect()

2022-02-26 23:35:38 Hail: INFO: Coerced sorted dataset
2022-02-26 23:35:38 Hail: INFO: reading 2 of 8 data partitions
2022-02-26 23:35:38 Hail: INFO: Coerced sorted dataset
2022-02-26 23:35:38 Hail: INFO: reading 3 of 8 data partitions
2022-02-26 23:35:39 Hail: INFO: Coerced sorted dataset
2022-02-26 23:35:39 Hail: INFO: reading 3 of 8 data partitions


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]

In [41]:
final

2022-02-26 23:30:13 Hail: INFO: Coerced sorted dataset
2022-02-26 23:30:13 Hail: INFO: reading 3 of 8 data partitions
2022-02-26 23:30:14 Hail: INFO: Coerced sorted dataset
2022-02-26 23:30:14 Hail: INFO: reading 4 of 8 data partitions
2022-02-26 23:30:14 Hail: INFO: Coerced sorted dataset
2022-02-26 23:30:14 Hail: INFO: reading 3 of 8 data partitions


In [43]:
mt.show()

2022-02-26 23:30:18 Hail: INFO: Coerced sorted dataset
2022-02-26 23:30:18 Hail: INFO: reading 3 of 8 data partitions
2022-02-26 23:30:18 Hail: INFO: Coerced sorted dataset
2022-02-26 23:30:18 Hail: INFO: reading 3 of 8 data partitions
2022-02-26 23:30:18 Hail: INFO: Coerced sorted dataset
2022-02-26 23:30:18 Hail: INFO: reading 4 of 8 data partitions
2022-02-26 23:30:19 Hail: INFO: Coerced sorted dataset
2022-02-26 23:30:19 Hail: INFO: reading 3 of 8 data partitions


,,,
,,0,1
locus,alleles,GT,GT
locus<GRCh38>,array<str>,call,call
chr1:1,"[""A"",""C""]",0/1,0/0
chr1:2,"[""A"",""C""]",1/1,0/1
chr1:3,"[""A"",""C""]",0/1,0/0
chr1:4,"[""A"",""C""]",0/1,0/1
chr1:5,"[""A"",""C""]",1/1,0/1
chr1:6,"[""A"",""C""]",0/0,0/0
chr1:7,"[""A"",""C""]",0/0,0/1


# Faster way to aggregate CH

In [ ]:
    mt = mt.annotate_entries(
        phase = (hl.case()
            .when(mt.GT == hl.parse_call("1|0"), 1)
            .when(mt.GT == hl.parse_call("0|1"), 2)
            .or_missing()
            )
    )

In [252]:
checkpoint = 'data/simulation/checkpoint/chr21_v100_s352_8ch.mt'
mt = hl.read_matrix_table(checkpoint)





def aggr_count_by_gene(mt: hl.MatrixTable, gene_expr):
    """Get aggregated phased/unphased. This is a faster method
    than using "collect_gt_by_expr", however, it does not allow
    for variant IDs to be collected.
    
    :param mt: MatrixTable with at least some phased entries
    :param gene_expr: StringExpression for what string to perform
    the aggregation on.
    
    """
    return (mt.group_rows_by(gene_symbol = gene_expr)
                .aggregate(
                    phased=hl.struct(
                        a1 = hl.agg.count_where(
                            (mt.GT == hl.parse_call("1|0"))),
                        a2 = hl.agg.count_where(
                            (mt.GT == hl.parse_call("0|1"))),
                        n = hl.agg.count_where(
                            (mt.GT.is_het_ref()) & (mt.GT.phased))
                    ),  
                    unphased=hl.struct(
                        n = hl.agg.count_where(
                            (mt.GT.is_het_ref()) & (~mt.GT.phased))
                    ),
                    hom_alt_n = hl.agg.count_where(mt.GT.is_hom_var())
                    )
           )

In [258]:
gene_expr = mt.consequence.vep.worst_csq_by_gene_canonical.gene_id
gene = aggr_count_by_gene(mt, gene_expr)
gene = gene.annotate_entries(
    p = calc_prob_ko(gene.hom_alt_n, gene.phased, gene.unphased)
)


In [265]:
checkpoint = 'data/simulation/checkpoint/chr21_v100_s352_8ch.mt'
mt = hl.read_matrix_table(checkpoint)

ma1 = mt.select_entries(mt.GT)
ma2 = mt.select_entries(mt.GT)

In [ ]:
ma1 = ma1.

AttributeError: MatrixTable instance has no field, method, or property 'hom_alt_n'
    Hint: use 'describe()' to show the names of all data fields.

In [227]:
def calc_prob_ko(hom_expr, phased_expr, unphased_expr):
    """Calculate probability of knockout based on phased 
       and unphased alleles (requires sum_gts_entries)
    
    :param hom_expr: integer for homozygous count 
    :param phased_expr: struct with integers a1 and a2
    :param unphased_expr: struct with integers a1 and a2
    """
    
    n = unphased_expr.n
    return (hl.case()
           .when(hom_expr > 0, 1) # homozygote
           .when(
               (phased_expr.a1 > 0) & # compound het
               (phased_expr.a2 > 0), 1)
           .when(
               ((phased_expr.a1 == 1) | # likely compound het (one phased het)
                (phased_expr.a2 == 1)) &
                (n > 0), 1 - (1 / 2) ** n)
           .when(
               ((phased_expr.a1 == 0) | # likely compound het (zero phased het)
                (phased_expr.a2 == 0)) &
                (n > 1), 1 - 2 * (1 / 2) ** n)
           .default(0)
            )

In [152]:
mt.filter_entries(mt.CH).entries().show()

2022-03-08 12:20:00 Hail: INFO: Coerced sorted dataset


,,,,
gene_symbol,s,n1,n2,CH
str,str,int64,int64,bool
"""ENSG00000160183""","""1536204""",2,2,True
"""ENSG00000160183""","""2785613""",1,2,True
"""ENSG00000160183""","""2802140""",2,1,True
"""ENSG00000160183""","""3496139""",3,1,True
"""ENSG00000160183""","""3849096""",3,1,True
"""ENSG00000160183""","""5756943""",1,1,True
"""ENSG00000160183""","""5768429""",1,3,True
"""ENSG00000160183""","""5914195""",1,2,True


In [269]:
def collect_phase_count_by_expr(mt: hl.MatrixTable, expr: hl.StringExpression):
    """Create a hail table of aggregated genotypes by expr

    :param mt: MatrixTable to be used
    :param expr: what expression to collapse on, e.g. "gene_id"
    """
    return mt.group_rows_by(expr).aggregate(
            gts=hl.agg.filter(mt.GT.is_non_ref(), hl.agg.collect(mt.GT)),
            varid=hl.agg.filter(mt.GT.is_non_ref(), hl.agg.collect(mt.varid))
            )

# Stable way to aggregate CH

In [14]:
checkpoint = 'data/simulation/checkpoint/chr21_v100_s352_8ch.mt'
mt = hl.read_matrix_table(checkpoint)


In [5]:
homs = mt.filter_entries(mt.GT.is_hom_var())


In [11]:
gene_expr = mt.consequence.vep.worst_csq_by_gene_canonical.gene_id
mt = mt.group_rows_by(
        gene_symbol = gene_expr,
        ).aggregate_entries(n1 = hl.agg.count_where(mt.GT == hl.parse_call("1|0")))
                    
                    

In [15]:
r = ko.aggr_count_calls(mt)



(44, 39)